Task 1: Data Exploration and Advanced Preprocessing

Step 1: Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score


Step 2: Load and Explore Dataset

In [2]:
# Load dataset
data = pd.read_csv("insurance.csv")
data.head()

# Basic info
data.info()
data.describe()

# Pairplot
sns.pairplot(data)
plt.show()

# Correlation heatmap
plt.figure(figsize=(8,6))
sns.heatmap(data.corr(), annot=True, cmap='coolwarm')
plt.show()

# Boxplots to detect outliers
for col in ['age','bmi','charges']:
    sns.boxplot(x=data[col])
    plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'insurance.csv'

Step 3: Data Cleaning

In [ ]:
# Handle missing values
data.fillna({'age': data['age'].median(),
             'bmi': data['bmi'].median(),
             'children': data['children'].mode()[0],
             'sex': data['sex'].mode()[0],
             'smoker': data['smoker'].mode()[0],
             'region': data['region'].mode()[0]}, inplace=True)

# Remove duplicates
data = data.drop_duplicates()

# Treat outliers using IQR
Q1 = data['bmi'].quantile(0.25)
Q3 = data['bmi'].quantile(0.75)
IQR = Q3 - Q1
data = data[(data['bmi'] >= Q1 - 1.5*IQR) & (data['bmi'] <= Q3 + 1.5*IQR)]


Step 4: Feature Engineering

In [ ]:
# BMI * Age interaction
data['BMI_Age_Interaction'] = data['bmi'] * data['age']

# Binary feature for young people
data['is_young'] = (data['age'] < 30).astype(int)

# Optional log-transform for charges
data['log_charges'] = np.log1p(data['charges'])


Step 5: Encoding and Scaling Experiments

In [ ]:
# Label Encoding
le = LabelEncoder()
data['sex_le'] = le.fit_transform(data['sex'])
data['smoker_le'] = le.fit_transform(data['smoker'])
data['region_le'] = le.fit_transform(data['region'])

# One-Hot Encoding
data = pd.get_dummies(data, columns=['sex','smoker','region'], drop_first=True)

# Scaling
scaler_std = StandardScaler()
scaled_features_std = scaler_std.fit_transform(data[['age','bmi','children','BMI_Age_Interaction']])

scaler_mm = MinMaxScaler()
scaled_features_mm = scaler_mm.fit_transform(data[['age','bmi','children','BMI_Age_Interaction']])


Task 2: Linear Regression Model and Parameter Tuning

Step 1: Split Dataset

In [ ]:
X = data.drop(columns=['charges','log_charges'])
y = data['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Step 2: Train and Evaluate Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

# Metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse, "RMSE:", rmse, "R²:", r2)

# Scatter plot
plt.scatter(y_test, y_pred)
plt.xlabel("Actual Charges")
plt.ylabel("Predicted Charges")
plt.title("Actual vs Predicted Charges")
plt.show()


Step 3: Regularized Models

In [ ]:
alphas = [0.01, 0.1, 1, 10, 100]
ridge_rmse, ridge_r2 = [], []

for a in alphas:
    model = Ridge(alpha=a)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    ridge_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))
    ridge_r2.append(r2_score(y_test, y_pred))

plt.plot(alphas, ridge_r2, marker='o')
plt.xlabel("Alpha")
plt.ylabel("R²")
plt.title("R² vs Alpha for Ridge Regression")
plt.show()


Step 4: Feature Importance

In [ ]:
coefs = pd.Series(lr.coef_, index=X.columns)
top_features = coefs.abs().sort_values(ascending=False).head(5)
print(top_features)


Task 4: Optimization and Cross-Validation

In [ ]:
# k-Fold CV
for model in [LinearRegression(), Ridge(alpha=1), Lasso(alpha=0.1)]:
    scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    print(model, "Average R²:", scores.mean())

# GridSearchCV for Ridge
params = {'alpha':[0.01,0.1,1,10,100]}
grid = GridSearchCV(Ridge(), params, cv=5, scoring='r2')
grid.fit(X_train, y_train)
print(grid.best_params_, grid.best_score_)


In [ ]:
Task 5: Manual Gradient Descent for Salary Prediction

In [ ]:
Step 1: Load Dataset

In [ ]:
salary_data = pd.read_csv("Salary.csv")
X = salary_data['YearsExperience'].values
y = salary_data['Salary'].values


Step 2: Implement Gradient Descent

In [ ]:
def gradient_descent(X, y, lr=0.01, epochs=1000):
    m = 0  # slope
    b = 0  # intercept
    n = len(y)
    cost_history = []

    for _ in range(epochs):
        y_pred = m*X + b
        error = y_pred - y
        cost = (1/n) * np.sum(error**2)
        cost_history.append(cost)
        
        # Update parameters
        m -= lr * (2/n) * np.sum(error * X)
        b -= lr * (2/n) * np.sum(error)
        
    return m, b, cost_history

m, b, cost_hist = gradient_descent(X, y, lr=0.01, epochs=1000)
plt.plot(cost_hist)
plt.xlabel("Epochs")
plt.ylabel("Cost")
plt.title("Cost vs Epochs")
plt.show()

print("Slope:", m, "Intercept:", b)


Step 3: Experiment with Learning Rates

In [ ]:
for lr in [0.001, 0.01, 0.1]:
    _, _, cost_hist = gradient_descent(X, y, lr=lr, epochs=1000)
    plt.plot(cost_hist, label=f'lr={lr}')

plt.xlabel("Epochs")
plt.ylabel("Cost")
plt.legend()
plt.show()
